# Model Experiments — Ames House Prices

This notebook compares untuned candidate models on one reproducible split. Reusable training and evaluation logic remains in `src/models/`.

## Experiment contract

All candidates train on the same `X_train`/`y_train` partition and are evaluated on the same held-out test set. Their preprocessing is inside each pipeline, so imputers, encoders, and scalers learn only from training rows. No hyperparameters are tuned in this notebook.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import seaborn as sns

project_root = Path.cwd().resolve()
if not (project_root / 'data').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.data.preprocessing import load_processed_data
from src.models.evaluate import evaluate_models
from src.models.train import create_train_test_split, train_model_comparison_candidates

sns.set_theme(style='whitegrid', palette='deep')

## Train and evaluate candidates

The linear model is a transparent compact benchmark. The tree models receive the broader non-redundant feature set because they can naturally capture non-linear effects and interactions. This is an architectural choice, not hyperparameter tuning.

In [ ]:
data = load_processed_data(project_root / 'data/processed/train_clean.csv')
X_train, X_test, y_train, y_test = create_train_test_split(data)
models = train_model_comparison_candidates(X_train, y_train)
results = evaluate_models(models, X_test, y_test)
results.style.format({'mae': '${:,.0f}', 'rmse': '${:,.0f}', 'r2': '{:.3f}'})

## Error comparison

**What we are analyzing:** typical absolute error and large-error-sensitive RMSE for each candidate.

**Why it matters:** the best R² alone may still have an unacceptably high dollar error or a complexity cost that is not justified for the API.

In [ ]:
plot_data = results.melt(id_vars='Model', value_vars=['mae', 'rmse'], var_name='metric', value_name='dollars')
plt.figure(figsize=(10, 6))
sns.barplot(data=plot_data, x='Model', y='dollars', hue='metric')
plt.title('Prediction error by model (lower is better)')
plt.xlabel('Model')
plt.ylabel('Error in dollars')
plt.xticks(rotation=12)
plt.tight_layout()

## Final candidate evaluation

The tuned Gradient Boosting pipeline is selected from training-only cross-validation. We now evaluate it on the held-out test partition and inspect residuals rather than reporting aggregate metrics alone.

In [ ]:
from src.models.evaluate import build_residual_frame, calculate_regression_metrics, extract_feature_importances
from src.models.train import tune_gradient_boosting_model

search = tune_gradient_boosting_model(X_train, y_train)
final_model = search.best_estimator_
residuals = build_residual_frame(final_model, X_test, y_test)
calculate_regression_metrics(y_test, residuals['predicted_price'])

### Actual versus predicted price

**What we are analyzing:** agreement between test-set sale prices and predictions.

**Why it matters:** points near the diagonal are accurate. A consistent offset or widening spread would reveal bias or error growth that aggregate metrics can hide.

In [ ]:
limits = [residuals[['actual_price', 'predicted_price']].min().min(), residuals[['actual_price', 'predicted_price']].max().max()]
plt.figure(figsize=(7, 7))
sns.scatterplot(data=residuals, x='actual_price', y='predicted_price', alpha=0.7)
plt.plot(limits, limits, linestyle='--', color='black', label='Perfect prediction')
plt.title('Actual versus predicted sale price')
plt.xlabel('Actual price ($)')
plt.ylabel('Predicted price ($)')
plt.legend()
plt.tight_layout()

### Residual diagnostics

**What we are analyzing:** signed errors (`actual − predicted`) overall and by prediction size.

**Why it matters:** residuals should generally be centered near zero without a clear curve or funnel shape. Large systematic patterns point to missing features, target transformation needs, or model bias.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=residuals, x='residual', kde=True, ax=axes[0])
axes[0].axvline(0, color='black', linestyle='--')
axes[0].set_title('Residual distribution')
sns.scatterplot(data=residuals, x='predicted_price', y='residual', alpha=0.7, ax=axes[1])
axes[1].axhline(0, color='black', linestyle='--')
axes[1].set_title('Residuals versus predicted price')
plt.tight_layout()
residuals[['residual', 'absolute_error']].describe()

### Feature importance

**What we are analyzing:** how strongly the fitted Gradient Boosting trees use each transformed feature to reduce prediction error.

**Why it matters:** importance is not causal proof, but it helps check whether the model relies on plausible property attributes and guides future investigation.

In [ ]:
importances = extract_feature_importances(final_model).head(15).sort_values()
plt.figure(figsize=(9, 7))
sns.barplot(x=importances.values, y=importances.index, hue=importances.index, legend=False)
plt.title('Top 15 Gradient Boosting feature importances')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
importances.sort_values(ascending=False)

## Model-selection discussion

Use the table and chart to assess all of the following before selecting a tuning candidate:

- **Prediction error:** lower MAE and RMSE are preferred.
- **Generalization:** tuning uses training-only cross-validation; the final diagnostics use the fixed held-out split.
- **Complexity:** Gradient Boosting is more costly and less transparent than Linear Regression.
- **Weaknesses:** review the largest absolute residuals, especially expensive homes where dollar error is likely to be larger.

The next operational phase serializes this complete tuned pipeline, not a bare estimator.